In [1]:
# Importing Modules
import numpy as np
import pandas as pd
import itertools
from sklearn.model_selection import train_test_split
from src.utils.smiles2morganfp import MorganFP
from src.gnn_fp_utils.dataloader import loadData
from src.gnn_fp_utils.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_data = pd.read_csv("data/processed/train/ESOL.csv")

# Generate ESOL FP
esol_fp = MorganFP(esol_data["smiles"])
esol_fp["smiles"] = esol_fp.index
esol_fp = esol_fp.merge(esol_data, on="smiles")

######################## DATA-2 ##################################
# Loading RT data
rt_data = pd.read_csv("data/processed/train/RT.csv")

# Generate RT FP
rt_fp = MorganFP(rt_data["smiles"])
rt_fp["smiles"] = rt_fp.index
rt_fp = rt_fp.merge(rt_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_data = pd.read_csv("data/processed/train/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_fp = MorganFP(lipophilicity_data["smiles"])
lipophilicity_fp["smiles"] = lipophilicity_fp.index
lipophilicity_fp = lipophilicity_fp.merge(lipophilicity_data, on="smiles")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_data = pd.read_csv("data/processed/train/B3DB.csv")

# Generate B3DB FP
b3db_fp = MorganFP(b3db_data["smiles"])
b3db_fp["smiles"] = b3db_fp.index
b3db_fp = b3db_fp.merge(b3db_data, on="smiles")

In [3]:
# Function to run analysis
def RunGNN(data, dataName, modelName, h_dim, b_size, lr, d_out, wd):

    # Train test split
	train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

	# Loading dataset
	train_loader = loadData(train_data, batch_size=b_size)
	test_loader = loadData(test_data, batch_size=b_size)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "h_dim":[h_dim], "b_size":[b_size], "lr":[lr],
            "w_decay":[wd], "d_out":[d_out],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in models:
        for i in grid_search_list:
            temp_out.append(RunGNN(data_dict[dataName], dataName, modelName, i["h_dim"], i["b_size"], i["lr"], i["dropout"], i["weight_decay"]))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_GNN_FP_{dataName}.csv", quoting=False, index=False)

    # Best parameters
    for model in models:
        temp = final_df[final_df["Model"]==model]
        print(temp.sort_values(by=["RMSE"]).head(1),"\n")

In [5]:
# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

# Hyperparameter dict
hyperparams = {
    'lr': [0.001, 0.0005],
    'h_dim': [16, 32, 64],
    'b_size': [16, 32],
    'weight_decay': [1e-4, 1e-5], 
    'dropout': [0.2, 0.5] 
}

# Data
data_dict = {
    "ESOL": esol_fp, 
    "Lipophilicity": lipophilicity_fp,
    "RT": rt_fp,
    "B3DB": b3db_fp
}

# Grid search list
keys = hyperparams.keys()
values = hyperparams.values()
grid_search_list = [dict(zip(keys, combination)) for combination in itertools.product(*values)]

In [6]:
# Params Optim for ESOL
search_hyperparams("ESOL")

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
18  ESOL   GCN     64      16  0.001  0.00001    0.2  0.92  1.29 

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
69  ESOL   GAT     64      32  0.001   0.0001    0.5  0.92   1.3 

     Data Model  h_dim  b_size     lr  w_decay  d_out  MAE  RMSE
119  ESOL   GIN     64      32  0.001  0.00001    0.5  0.9  1.31 

     Data      Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
153  ESOL  GraphSAGE     32      16  0.001   0.0001    0.5  0.92  1.31 



In [7]:
# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

             Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
19  Lipophilicity   GCN     64      16  0.001  0.00001    0.5  0.75  0.96 

             Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
71  Lipophilicity   GAT     64      32  0.001  0.00001    0.5  0.77  0.98 

              Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
117  Lipophilicity   GIN     64      32  0.001   0.0001    0.5  0.74  0.96 

              Data      Model  h_dim  b_size     lr  w_decay  d_out   MAE  \
161  Lipophilicity  GraphSAGE     64      16  0.001   0.0001    0.5  0.75   

     RMSE  
161  0.97   



In [8]:
# Params Optim for RT
search_hyperparams("RT")

   Data Model  h_dim  b_size      lr  w_decay  d_out    MAE    RMSE
36   RT   GCN     32      32  0.0005   0.0001    0.2  72.47  104.57 

   Data Model  h_dim  b_size      lr  w_decay  d_out    MAE    RMSE
74   RT   GAT     16      16  0.0005  0.00001    0.2  72.98  103.96 

    Data Model  h_dim  b_size      lr  w_decay  d_out   MAE    RMSE
120   RT   GIN     16      16  0.0005   0.0001    0.2  73.9  103.88 

    Data      Model  h_dim  b_size      lr  w_decay  d_out    MAE    RMSE
180   RT  GraphSAGE     32      32  0.0005   0.0001    0.2  72.71  102.98 



In [9]:
# Params Optim for B3DB
search_hyperparams("B3DB")

   Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
3  B3DB   GCN     16      16  0.001  0.00001    0.5  0.36  0.51 

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
67  B3DB   GAT     64      16  0.001  0.00001    0.5  0.36  0.51 

     Data Model  h_dim  b_size      lr  w_decay  d_out   MAE  RMSE
143  B3DB   GIN     64      32  0.0005  0.00001    0.5  0.36  0.52 

     Data      Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
155  B3DB  GraphSAGE     32      16  0.001  0.00001    0.5  0.36  0.52 

